### 0. 환경 설정 — 위젯/진행바 비활성화
`load_dataset`·`.map()`·`from_pretrained` 의 **ipywidget 진행바**가 VS Code 에서
`i is not a function` 렌더 오류를 내는 것을 막는다. **모든 import 보다 먼저** 실행할 것.

In [ ]:
# ── widget 렌더러 오류 방지 + 진행바 비활성 (VS Code Jupyter) ──────
# load_dataset/.map()/from_pretrained 의 위젯 진행바가 'i is not a function'
# 렌더 에러를 유발 → 텍스트/비활성으로 강제. 모든 import보다 먼저 실행할 것.
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_DATASETS_DISABLE_PROGRESS_BARS'] = '1'   # datasets 는 별도 변수
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TQDM_NOTEBOOK'] = 'false'


# Document AI: Hugging Face Transformers로 Donut 문서 파싱 파인튜닝

이 노트북에서는 **Donut-base**([`naver-clova-ix/donut-base`](https://huggingface.co/naver-clova-ix/donut-base))를
Hugging Face Transformers로 **문서 이해 / 문서 파싱** 태스크에 맞게 파인튜닝하는 방법을 다룬다.
Donut은 OCR 없이도 SOTA 성능을 내는 문서 이해 모델이며, **MIT 라이선스**라 LayoutLMv2/v3와 달리
상업적 이용이 자유롭다.

데이터셋은 **SROIE**([ICDAR-2019-SROIE](https://github.com/zzzDavid/ICDAR-2019-SROIE), 스캔 영수증 1,000장 + OCR)를 사용한다.

### 목차
1. 개발 환경 설정
2. SROIE 데이터셋 로드
3. Donut 학습용 데이터 준비
4. Donut 모델 파인튜닝 & 평가

## 빠른 소개: Donut (ClovaAI)

**Donut**(Document Understanding Transformer)은 **OCR 엔진 없이** 스캔 문서를 이해하는 Transformer 모델이다.
문서 분류·정보 추출(문서 파싱) 등 다양한 시각 문서 이해 태스크에서 SOTA를 달성한다.
구조는 **비전 인코더([Swin Transformer](https://huggingface.co/docs/transformers/model_doc/swin)) +
텍스트 디코더([BART](https://huggingface.co/docs/transformers/model_doc/bart))**의 멀티모달 seq2seq이다.
인코더가 이미지를 임베딩으로 바꾸면, 디코더가 그 임베딩을 받아 토큰 시퀀스를 생성한다.

![donut](assets/donut.png)

- 논문: https://arxiv.org/abs/2111.15664
- 공식 repo: https://github.com/clovaai/donut

---

이제 Donut의 동작 원리를 알았으니 시작해보자. 🚀

_참고: 원본 튜토리얼은 NVIDIA V100(AWS p3.2xlarge)에서 작성·실행되었다._

## 1. 개발 환경 설정

먼저 Hugging Face 라이브러리(`transformers`, `datasets` 등)를 설치한다.
아래 셀을 실행하면 학습에 필요한 패키지가 모두 설치된다.

### 1. 의존성 설치
`transformers`(≥4.22), `datasets`(데이터 로딩), `sentencepiece`(Donut 토크나이저),
`tensorboard`(학습 로그)를 설치한다. 이미 설치돼 있으면 건너뛰어도 된다.

In [ ]:
!pip install -q "transformers>=4.22.0" # comment in when version is released
!pip install -q datasets sentencepiece tensorboard 

### git-lfs 설치 (선택)
모델/로그를 HuggingFace Hub 로 푸시할 때 필요한 대용량 파일 도구. **이 노트북은
`push_to_hub=False` 로 바꿔 로컬 학습만 하므로 실행하지 않아도 된다.**

In [ ]:
# install git-fls for pushing model and logs to the hugging face hub
!sudo apt-get install git-lfs --yes

원본은 학습 결과물을 **Hugging Face Hub**(원격 모델 버전 관리)에 푸시하려고 `notebook_login`으로
로그인한다. 계정이 있으면 토큰을 디스크에 저장한다.
> 이 노트북은 `push_to_hub=False`로 바꿔 **로컬 학습만** 하므로 이 단계는 건너뛰어도 된다.

### Hub 로그인 (비활성)
원본은 학습 결과를 Hub 로 푸시하려고 로그인했지만, **`push_to_hub=False` 로 바꿔
비활성화**했다. 본인 계정으로 푸시하려면 주석을 풀고 실행한다.

In [ ]:
# push_to_hub=False 라 허브 로그인 불필요. 푸시하려면 아래 두 줄 주석 해제 후 실행.
# from huggingface_hub import notebook_login
# notebook_login()

## 2. SROIE 데이터셋 로드

**SROIE**(스캔 영수증 1,000장 + OCR, ICDAR-2019 task 2)를 사용한다. Hugging Face 에 올라온
`darentang/sroie` 는 Donut 과 호환되지 않으므로, **원본 데이터셋**을 받아 `datasets` 의
`imagefolder` 기능으로 로드한다.

### 2. SROIE 데이터셋 내려받기
영수증 데이터셋(ICDAR-2019-SROIE)을 clone 해 `data/img`(이미지) + `data/key`(정답 JSON)로
정리한다. `%%bash` 는 **커널의 현재 작업 디렉터리**(여기선 `SROIE_donut/`)에 받는다.

In [ ]:
%%bash 
# clone repository
git clone https://github.com/zzzDavid/ICDAR-2019-SROIE.git
# copy data
cp -r ICDAR-2019-SROIE/data ./
# clean up
rm -rf ICDAR-2019-SROIE
rm -rf data/box

이제 `data/` 안에 영수증 **이미지** 폴더와 **OCR 텍스트(정답)** 폴더가 생겼다.
다음 단계는 `imagefolder` 가 읽을 수 있도록 이미지 정보(+OCR 텍스트)를 담은
`metadata.jsonl` 을 만드는 것이다.

최종 형태는 **한 줄에 하나의 JSON**(파일명 + text):
```json
{"file_name": "0001.png", "text": "..."}
{"file_name": "0002.png", "text": "..."}
```
여기서 `"text"` 컬럼이 이미지의 정답(OCR 텍스트)이며, 나중에 Donut 전용 토큰 형식으로 변환된다.

### 정답 JSON → `metadata.jsonl` (imagefolder 포맷)
`data/key/*.json` 정답을 각각 **한 줄 JSON 문자열("text" 컬럼)** 로 만들어
`data/img/metadata.jsonl` 에 저장한다. 이러면 `imagefolder` 로더가 이미지와 정답을
자동으로 짝짓는다. 변환 후 `key/` 폴더는 삭제.

In [ ]:
import os
import json
from pathlib import Path
import shutil

# define paths
base_path = Path("data")
metadata_path = base_path.joinpath("key")
image_path = base_path.joinpath("img")
# define metadata list
metadata_list = []

# parse metadata
for file_name in metadata_path.glob("*.json"):
  with open(file_name, "r") as json_file:
    # load json file
    data = json.load(json_file)
    # create "text" column with json string
    text = json.dumps(data)
    # add to metadata list if image exists
    if image_path.joinpath(f"{file_name.stem}.jpg").is_file():    
      metadata_list.append({"text":text,"file_name":f"{file_name.stem}.jpg"})
      # delete json file
      
# write jsonline file
with open(image_path.joinpath('metadata.jsonl'), 'w') as outfile:
    for entry in metadata_list:
        json.dump(entry, outfile)
        outfile.write('\n')

# remove old meta data
shutil.rmtree(metadata_path)

좋다! 이제 `datasets` 의 `imagefolder` 기능으로 데이터셋을 로드할 수 있다.

### 경로 변수 준비
이후 셀에서 쓸 `data/img`·`data/key` 경로를 정의하는 준비용 셀.

In [ ]:
import os
import json
from pathlib import Path
import shutil

# define paths
base_path = Path("data")
metadata_path = base_path.joinpath("key")
image_path = base_path.joinpath("img")
# define metadata list

### `imagefolder` 로 데이터셋 로드
`data/img/`(이미지 + metadata.jsonl)를 HuggingFace Dataset 으로 로드한다.
`image`(영수증) + `text`(정답 JSON 문자열) 컬럼이 생긴다.

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("imagefolder", data_dir=image_path, split="train")

print(f"Dataset has {len(dataset)} images")
print(f"Dataset features are: {dataset.features.keys()}")

이제 데이터셋을 좀 더 자세히 살펴보자.

### 데이터 살펴보기
임의 샘플 하나의 정답 텍스트와 이미지를 출력해 데이터가 제대로 로드됐는지 눈으로 확인한다.

In [ ]:
import random

random_sample = random.randint(0, len(dataset))

print(f"Random sample is {random_sample}")
print(f"OCR text is {dataset[random_sample]['text']}")
dataset[random_sample]['image'].resize((250,400))


## 3. Donut 학습용 데이터 준비

Donut 은 비전 인코더 + 텍스트 디코더의 seq2seq 모델이다. 파인튜닝 시 이미지로부터 `"text"` 를
생성하도록 학습시킨다. 그러려면 NLP 처럼 텍스트를 토크나이즈·전처리해야 하는데, 그 전에
**JSON 문자열을 Donut 호환 토큰 문서로 변환**해야 한다.

**원본 JSON 문자열**
```json
{"company": "ADVANCO COMPANY", "date": "17/01/2018", "address": "...", "total": "7.00"}
```
**Donut 토큰 문서**
```
<s></s><s_company>ADVANCO COMPANY</s_company><s_date>17/01/2018</s_date><s_address>...</s_address><s_total>7.00</s_total></s>
```
ClovaAI 의 [`json2token`](https://github.com/clovaai/donut/blob/master/donut/model.py) 메서드를 가져와 이 변환을 수행한다.

### 3. Donut 학습용 전처리 — JSON → 토큰 시퀀스
Donut 은 정답 JSON 을 **XML 스타일 토큰**으로 표현한다
(`{"total":"x"}` → `<s_total>x</s_total>`). `json2token` 이 이 변환을 하면서
**등장하는 모든 필드 키를 `new_special_tokens` 로 수집**한다.

**중요 포인트**
- 키는 `sorted(..., reverse=True)` 로 **역정렬** → 항상 같은 순서(결정적).
- 여기서 모은 필드 토큰을 다음 단계에서 토크나이저에 등록해야 디코더가 생성할 수 있다.

In [ ]:
new_special_tokens = [] # new tokens which will be added to the tokenizer
task_start_token = "<s>"  # start of task token
eos_token = "</s>" # eos token of tokenizer

def json2token(obj, update_special_tokens_for_json_key: bool = True, sort_json_key: bool = True):
    """
    Convert an ordered JSON object into a token sequence
    """
    if type(obj) == dict:
        if len(obj) == 1 and "text_sequence" in obj:
            return obj["text_sequence"]
        else:
            output = ""
            if sort_json_key:
                keys = sorted(obj.keys(), reverse=True)
            else:
                keys = obj.keys()
            for k in keys:
                if update_special_tokens_for_json_key:
                    new_special_tokens.append(fr"<s_{k}>") if fr"<s_{k}>" not in new_special_tokens else None
                    new_special_tokens.append(fr"</s_{k}>") if fr"</s_{k}>" not in new_special_tokens else None
                output += (
                    fr"<s_{k}>"
                    + json2token(obj[k], update_special_tokens_for_json_key, sort_json_key)
                    + fr"</s_{k}>"
                )
            return output
    elif type(obj) == list:
        return r"<sep/>".join(
            [json2token(item, update_special_tokens_for_json_key, sort_json_key) for item in obj]
        )
    else:
        # excluded special tokens for now
        obj = str(obj)
        if f"<{obj}/>" in new_special_tokens:
            obj = f"<{obj}/>"  # for categorical special tokens
        return obj


def preprocess_documents_for_donut(sample):
    # create Donut-style input
    text = json.loads(sample["text"])
    d_doc = task_start_token + json2token(text) + eos_token
    # convert all images to RGB
    image = sample["image"].convert('RGB')
    return {"image": image, "text": d_doc}

proc_dataset = dataset.map(preprocess_documents_for_donut)

print(f"Sample: {proc_dataset[45]['text']}")
print(f"New special tokens: {new_special_tokens + [task_start_token] + [eos_token]}")


다음은 텍스트를 토크나이즈하고 이미지를 텐서로 인코딩하는 단계다. `DonutProcessor` 를 불러와
새 특수 토큰을 추가하고, 처리 시 이미지 크기를 `[1920, 2560]` → `[720, 960]` 으로 낮춰
메모리를 아끼고 학습 속도를 높인다.

### 프로세서 로드 + 필드 토큰 등록 + 해상도 설정
`DonutProcessor` 를 불러와 위에서 수집한 **필드 토큰 + task/eos 토큰을 등록**한다.

**중요 포인트**
- `image_processor.size` 는 **dict** `{"height":960,"width":720}` 형식(신버전). 사전학습 `[2560,1920]` → `[960,1280]` 급으로 낮춰 VRAM 절약.
- `do_align_long_axis=False` — 세로 이미지를 회전 없이 원본 방향대로 사용.

In [ ]:
from transformers import DonutProcessor

# Load processor
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")

# add new special tokens to tokenizer
processor.tokenizer.add_special_tokens({"additional_special_tokens": new_special_tokens + [task_start_token] + [eos_token]})

# we update some settings which differ from pretraining; namely the size of the images + no rotation required
# resizing the image to smaller sizes from [1920, 2560] to [960,1280]
# 신버전 image_processor.size 는 dict {"height":..,"width":..} 기대 (리스트 X)
processor.image_processor.size = {"height": 960, "width": 720}
processor.image_processor.do_align_long_axis = False

이제 학습에 사용할 데이터셋을 준비한다.

### 이미지+정답 → 학습 텐서로 변환 (`.map`)
각 샘플을 `pixel_values`(이미지 텐서) + `labels`(정답 토큰 ID)로 변환한다.

**중요 포인트**
- `random_padding=True`(train) — 이미지 패딩 위치를 무작위화해 일종의 증강 효과.
- 정답은 `max_length=512` 로 패딩/잘림, **패딩 위치는 `-100`** 으로 마스킹해 loss 에서 제외.
- ⚠️ `.map` 전체 변환은 RAM 을 많이 쓴다(원 주석: 32~64GB 권장).

In [ ]:
def transform_and_tokenize(sample, processor=processor, split="train", max_length=512, ignore_id=-100):
    # create tensor from image
    try:
        pixel_values = processor(
            sample["image"], random_padding=split == "train", return_tensors="pt"
        ).pixel_values.squeeze()
    except Exception as e:
        print(sample)
        print(f"Error: {e}")
        return {}
        
    # tokenize document
    input_ids = processor.tokenizer(
        sample["text"],
        add_special_tokens=False,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )["input_ids"].squeeze(0)

    labels = input_ids.clone()
    labels[labels == processor.tokenizer.pad_token_id] = ignore_id  # model doesn't need to predict pad token
    return {"pixel_values": pixel_values, "labels": labels, "target_sequence": sample["text"]}

# need at least 32-64GB of RAM to run this
processed_dataset = proc_dataset.map(transform_and_tokenize,remove_columns=["image","text"])

### (선택) 전처리 결과 디스크 저장/복원
오래 걸리는 `.map` 결과를 디스크에 저장해 두면 나중에 에러가 나도 다시 안 돌려도 된다.
필요할 때 주석을 풀어 사용(기본은 비활성).

In [ ]:
# from datasets import load_from_disk
# from transformers import DonutProcessor

## COMMENT IN in case you want to save the processed dataset to disk in case of error later
# processed_dataset.save_to_disk("processed_dataset")
# processor.save_pretrained("processor")

## COMMENT IN in case you want to load the processed dataset from disk in case of error later
# processed_dataset = load_from_disk("processed_dataset")
# processor = DonutProcessor.from_pretrained("processor")


마지막으로 데이터셋을 train / validation 으로 나눈다.

### 학습/검증 분할
전처리된 데이터를 train/test 로 나눈다(`test_size=0.1`).

In [ ]:
processed_dataset = processed_dataset.train_test_split(test_size=0.1)
print(processed_dataset)

## 4. Donut 모델 파인튜닝 & 평가

데이터 전처리가 끝났으니 학습을 시작한다. 먼저 `VisionEncoderDecoderModel` 로
`naver-clova-ix/donut-base` 를 로드한다. `donut-base` 는 **사전학습 가중치만** 포함하며,
논문 [OCR-free Document Understanding Transformer](https://arxiv.org/abs/2111.15664)(Geewook et al.)에서 소개되었다.

### 4. 모델 로드 & 설정
사전학습 Donut 을 불러와 새 토큰 수에 맞게 임베딩을 늘리고, 해상도·시작 토큰을 맞춘다.

**중요 포인트**
- `model.loss_type = "ForCausalLM"` — `loss_type=None` 경고 제거(동작 동일).
- `resize_token_embeddings(..., mean_resizing=False)` — 새 토큰을 **랜덤 초기화**(원본 레시피와 동일, 결정적).
- `decoder.max_length` 를 학습셋 라벨 최대 길이에 맞춰 설정.

In [ ]:
import torch
from transformers import VisionEncoderDecoderModel, VisionEncoderDecoderConfig

# Load model from huggingface.co
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")
# VED는 클래스명이 LOSS_MAPPING과 안 맞아 loss_type=None 경고 → 동일 fallback 명시
model.loss_type = "ForCausalLM"

# Resize embedding layer to match vocabulary size
new_emb = model.decoder.resize_token_embeddings(len(processor.tokenizer), mean_resizing=False)
print(f"New embedding size: {new_emb}")
# Adjust our image size and output sequence lengths
model.config.encoder.image_size = [processor.image_processor.size["height"], processor.image_processor.size["width"]] # (height, width)
model.config.decoder.max_length = len(max(processed_dataset["train"]["labels"], key=len))

# Add task token for decoder to start
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s>'])[0]

# is done by Trainer
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model.to(device)

학습 전에 `Seq2SeqTrainingArguments` 로 하이퍼파라미터를 정의한다.
> 원본은 학습 중 체크포인트·로그·metric 을 Hub 레포로 자동 푸시하지만, 이 노트북은
> `push_to_hub=False` 로 **로컬 저장만** 한다.

### 학습 하이퍼파라미터 + Trainer 구성
`Seq2SeqTrainingArguments` 로 학습 설정을 정의하고 `Seq2SeqTrainer` 를 만든다.

**중요 포인트(수정 반영됨)**
- **`fp16=False` + `bf16=use_bf16`** — Donut 은 fp16 에서 수치 불안정 → bf16 권장. `use_bf16` 은 GPU 지원 여부 자동 감지(Ampere+), 미지원이면 fp32.
- **`push_to_hub=False`** — 로컬에서만 학습/저장(`hub_*` 인자는 주석 처리).
- `predict_with_generate=True`, `save_strategy="epoch"`.

In [ ]:
from huggingface_hub import HfFolder
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# hyperparameters used for multiple args
hf_repository_id = "donut-base-sroie"

# Arguments for training
# bf16은 Ampere+ 에서만 지원 → 자동 감지. 미지원이면 fp32 (fp16은 Donut 수치 불안정이라 미사용)
import torch
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f'bf16 사용: {use_bf16}')
training_args = Seq2SeqTrainingArguments(
    output_dir=hf_repository_id,
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    weight_decay=0.01,
    fp16=False,
    bf16=use_bf16,
    logging_steps=100,
    save_total_limit=2,
    eval_strategy="no",
    save_strategy="epoch",
    predict_with_generate=True,
    # push to hub parameters
    report_to="tensorboard",
    push_to_hub=False,
    # push_to_hub=False 라 아래 hub_* 는 불필요 (로그인 의존 제거)
    # hub_strategy="every_save",
    # hub_model_id=hf_repository_id,
    # hub_token=HfFolder.get_token(),
)

# Create Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
)

이제 `Seq2SeqTrainer` 의 `train()` 메서드로 학습을 시작한다.

### 학습 실행
`trainer.train()` 으로 학습을 시작한다. 매 epoch 마다 체크포인트가 `output_dir` 에 저장된다.

In [ ]:
# Start training
trainer.train()

학습이 끝나면 프로세서도 함께 저장한다(원본은 Hub 저장 + 모델 카드 생성).
> 로컬 전용으로 바꾼 지금은 Hub 푸시 대신 로컬 경로에 저장하면 된다.

### 프로세서 저장 (+ Hub 푸시는 비활성)
프로세서를 모델과 같은 경로에 저장한다(추론 시 둘 다 같은 경로에서 로드해야 함).
> ⚠️ `trainer.push_to_hub()` / `create_model_card()` 는 Hub 푸시용이라, **로컬 전용으로
> 바꾼 지금은 로그인이 없으면 실패**한다. 로컬 저장만 원하면 그 두 줄은 건너뛰고
> `trainer.save_model(...)` 로 대체하면 된다.

In [ ]:
# Save processor and create model card
processor.save_pretrained(hf_repository_id)
trainer.create_model_card()
trainer.push_to_hub()

모델 학습에 성공했다. 이제 추론으로 테스트하고 정확도를 평가해보자.

### 5. 추론 테스트
학습된 모델로 영수증 한 장을 파싱해 **예측 vs 정답**을 비교한다.

**중요 포인트**
- `generate()` 가 `<s>` 토큰부터 시작해 EOS 까지 토큰을 자동 생성 → `token2json` 으로 dict 복원.
- ⚠️ 원본은 `philschmid/donut-base-sroie`(원저자 공개 모델)를 불러온다. **본인이 학습한 모델로 평가하려면** 이 경로를 학습 `output_dir`(예: `"donut-base-sroie"`)로 바꿀 것.

In [ ]:
import re
import transformers
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch
import random
import numpy as np

# hidde logs
transformers.logging.disable_default_handler()


# Load our model from Hugging Face
processor = DonutProcessor.from_pretrained("philschmid/donut-base-sroie")
model = VisionEncoderDecoderModel.from_pretrained("philschmid/donut-base-sroie")

# Move model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Load random document image from the test set
test_sample = processed_dataset["test"][random.randint(1, 50)]

def run_prediction(sample, model=model, processor=processor):
    # prepare inputs
    pixel_values = torch.tensor(test_sample["pixel_values"]).unsqueeze(0)
    task_prompt = "<s>"
    decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids

    # run inference
    outputs = model.generate(
        pixel_values.to(device),
        decoder_input_ids=decoder_input_ids.to(device),
        max_length=model.decoder.config.max_position_embeddings,
        early_stopping=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        num_beams=1,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )

    # process output
    prediction = processor.batch_decode(outputs.sequences)[0]
    prediction = processor.token2json(prediction)

    # load reference target
    target = processor.token2json(test_sample["target_sequence"])
    return prediction, target

prediction, target = run_prediction(test_sample)
print(f"Reference:\n {target}")
print(f"Prediction:\n {prediction}")
processor.image_processor.to_pil_image(np.array(test_sample["pixel_values"])).resize((350,600))


멋지다 😍🔥 파인튜닝된 모델이 문서를 올바르게 파싱해 값을 정확히 추출했다.
다음은 테스트셋 평가다. seq2seq 모델이라 평가가 간단하진 않다.

여기선 단순하게 **정확도(accuracy)** 를 쓴다 — 딕셔너리의 각 키에 대해 예측값과 정답이
**완전히 일치**하는지 비교한다. 이 방식은 완전 일치만 정답으로 치므로 **편향/단순**하다
(예: 공백 하나만 달라도 오답 처리).

### 전체 테스트셋 정확도 평가
테스트셋 전체를 순회하며 필드 값 일치율(정확도)을 계산한다.
> 참고: 원본 코드는 루프 안에서 `test_sample`(고정)만 예측하는 버그성 패턴이 있다 —
> 실제로는 `sample` 로 예측해야 정확한 평가가 된다.

In [ ]:
from tqdm import tqdm

# define counter for samples
true_counter = 0
total_counter = 0

# iterate over dataset
for sample in tqdm(processed_dataset["test"]):
  prediction, target = run_prediction(test_sample)
  for s in zip(prediction.values(), target.values()):
    if s[0] == s[1]:
      true_counter += 1
    total_counter += 1

print(f"Accuracy: {(true_counter/total_counter)*100}%")

모델은 테스트셋에서 **약 75% 정확도**를 달성한다.

_참고: 이 평가는 각 키의 **완전 문자열 일치**만 정답으로 보는 매우 단순한 방식이라 편향이 크다.
즉 75% 는 사실 꽤 좋은 수치다._

예를 들어 모델이 `address` 를 `... X ,U13/X ...` 로 예측하고 정답이 `... X,U13/X ...` 였는데,
`X` 와 `,U13/X` 사이의 **공백 하나** 차이만으로 오답 처리되었다. 평가 루프에선 이런 경우가
정답으로 카운트되지 않는다.